In [52]:
import pandas as pd
import numpy as np
from scipy.constants import pi

branch_data = pd.read_csv("branch.csv")
h_11 = 5 * 50

def create_z_11(branch_data):
    return [
        complex(row['r'], row['l'] * 2 * pi * h_11)
        for _, row in branch_data.iterrows()
    ]

def create_y_11(branch_data):
    return [
        complex(0, row['c'] * 2 * pi * h_11)
        for _, row in branch_data.iterrows()
    ]

def create_gammaL_11(branch_data, z_11, y_11):
    z_11 = np.array(z_11)
    y_11 = np.array(y_11)
    gammaL_11 = []
    for i, row in branch_data.iterrows():
        length = row['Length']
        gammaL = length * np.sqrt(z_11[i] * y_11[i])
        gammaL_11.append(gammaL)
    return gammaL_11

z_11 = create_z_11(branch_data)
y_11 = create_y_11(branch_data)
gammaL_11 = create_gammaL_11(branch_data, z_11, y_11)

def Z0_11(z_11, y_11):
    return np.sqrt(np.array(z_11) / np.array(y_11))

Z0_11 = Z0_11(z_11, y_11)
def ABCD_11(gammaL_11, Z0_11):
    A = np.cosh(gammaL_11)
    B = Z0_11 * np.sinh(gammaL_11)
    C = (1 / Z0_11) * np.sinh(gammaL_11)
    D = np.cosh(gammaL_11)
    return A, B, C, D
A_11, B_11, C_11, D_11 = ABCD_11(gammaL_11, Z0_11)
ABCD_11_df = pd.DataFrame({
    'A_11': A_11,
    'B_11': B_11,
    'C_11': C_11,
    'D_11': D_11
})
ABCD_11_df.to_csv("ABCD_11.csv", index=False)

In [57]:
import pandas as pd
import numpy as np

# Load and ensure numeric types
ABCD = pd.read_csv("ABCD_11.csv")

# Clean column names (optional, in case there are hidden spaces)
ABCD.columns = ABCD.columns.str.strip()

# Convert to complex numbers safely
for col in ['A_11', 'B_11', 'C_11', 'D_11']:
    ABCD[col] = ABCD[col].apply(lambda x: complex(x.replace('i', 'j')) if isinstance(x, str) else complex(x))

def combine_parallel(group):
    if len(group) == 1:
        row = group.iloc[0]
        return pd.Series({
            'From Bus Number': row['From Bus  Number'],
            'To Bus Number': row['To Bus  Number'],
            'A': row['A_11'],
            'B': row['B_11'],
            'C': row['C_11'],
            'D': row['D_11']
        })

    # Start with first line
    A_eq, B_eq, C_eq, D_eq = group.iloc[0][['A_11', 'B_11', 'C_11', 'D_11']]

    for _, row in group.iloc[1:].iterrows():
        A1, B1, C1, D1 = A_eq, B_eq, C_eq, D_eq
        A2, B2, C2, D2 = row['A_11'], row['B_11'], row['C_11'], row['D_11']

        A_eq = (A1 * B2 + A2 * B1) / (B1 + B2)
        B_eq = (B1 * B2) / (B1 + B2)
        C_eq = C1 + C2 + ((A1 - A2) * (D1 - D2)) / (B1 + B2)
        D_eq = (D1 * B2 + D2 * B1) / (B1 + B2)

    return pd.Series({
        'From Bus Number': group.iloc[0]['From Bus  Number'],
        'To Bus Number': group.iloc[0]['To Bus  Number'],
        'A': A_eq, 'B': B_eq, 'C': C_eq, 'D': D_eq
    })

# Combine lines with same (From, To)
ABCD_combined = (
    ABCD.groupby(['From Bus  Number', 'To Bus  Number'])
    .apply(combine_parallel)
    .reset_index(drop=True)
)

# Save result
ABCD_combined.to_csv("ABCD_parallel_combined.csv", index=False)

print(ABCD_combined.head())


   From Bus Number   To Bus Number                   A                    B  \
0   2135.0+   0.0j  2220.0+   0.0j  0.930616+0.002070j  0.145142+10.087491j   
1   2135.0+   0.0j  2400.0+   0.0j -0.018558+0.030480j -0.026429+47.055113j   
2   2135.0+   0.0j  2970.0+   0.0j -0.226617-0.016158j -0.110736-28.628424j   
3   2220.0+   0.0j  2225.0+   0.0j  0.957358-0.007798j  0.336360- 3.796587j   
4   2220.0+   0.0j  2230.0+   0.0j -0.007088-0.012550j -0.002380-22.755487j   

                    C                   D  
0  0.000191+0.013282j  0.930616+0.002070j  
1 -0.000012+0.021264j -0.018558+0.030480j  
2 -0.000128-0.033146j -0.226617-0.016158j  
3  0.001968-0.022175j  0.957358-0.007798j  
4 -0.000003-0.043950j -0.007088-0.012550j  


C:\Users\User\AppData\Local\Temp\ipykernel_7404\2143568679.py:47: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(combine_parallel)
